# 1. Versioned Legal Knowledge Base & Section-Wise Statutory Matrix
**Legal Knowledge RAG Engine — a local engineering prototype**

This project demonstrates a simple rule-based legal document assistant with local section search and review checks. It retrieves legal context but does not use a language model to generate answers, so it is not a full RAG pipeline.

Run all cells in order in a standard Colab CPU runtime. Only Python and the preinstalled NumPy/scikit-learn stack are used. No downloads, credentials, GPU, external services or network requests are needed. Retrieval and review are local and extractive; there is no generative LLM.

A complete statutory section is the indexing boundary: arbitrary character cuts can detach exceptions, conditions and provisos from the rule they qualify. Prefixing Act metadata distinguishes identical section numbers across Acts. **The eight records below are section-level educational summaries, not complete official statutory text.** For real use, replace each summary with the complete, verified section and its provisos from the official Laws of Bangladesh portal (https://bdlaws.minlaw.gov.bd), retaining the same boundary.

Pleading modules need immutable versions because changing a prerequisite, citation or wording can change a legal strategy. Each output records its version, effective snapshot date and content fingerprint. All case facts are fictional. The judgment update is a **synthetic High Court fixture**, not a claim about a recent real judgment or an amendment. Court interpretation and legislative amendment are distinct. This prototype is not filing-ready legal advice.

In [1]:
from copy import deepcopy
from dataclasses import dataclass
import hashlib, json, re
from datetime import date
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def table(rows, columns):
    """Plain text tables remain readable in Colab and exported notebooks."""
    values = [[str(r.get(c, '')) for c in columns] for r in rows]
    widths = [max([len(c)] + [len(r[i]) for r in values]) for i,c in enumerate(columns)]
    print(' | '.join(c.ljust(w) for c,w in zip(columns,widths)))
    print('-+-'.join('-'*w for w in widths))
    for row in values: print(' | '.join(v.ljust(w) for v,w in zip(row,widths)))

def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()

SOURCE_ROWS = [
 ('CRPC_54','Code of Criminal Procedure, 1898','54',
  'Arrest without warrant is subject to statutory grounds and safeguards, and the factual basis for arrest must be examined.'),
 ('CRPC_167','Code of Criminal Procedure, 1898','167',
  'Where investigation cannot be completed within twenty four hours, the accused and relevant diary entries are forwarded to a Magistrate; further detention requires judicial authorization subject to statutory limits.'),
 ('CRPC_496','Code of Criminal Procedure, 1898','496',
  'A person accused of a bailable offence who is prepared to give bail shall be released on bail, subject to the section.'),
 ('CRPC_497','Code of Criminal Procedure, 1898','497',
  'Bail in non bailable offences is discretionary and subject to statutory restrictions and exceptions; further inquiry is a distinct statutory consideration.'),
 ('PC_379','Penal Code, 1860','379',
  'Theft is punishable with imprisonment which may extend to three years, or with fine, or with both.'),
 ('PC_420','Penal Code, 1860','420',
  'Cheating and dishonestly inducing delivery of property is punishable with imprisonment which may extend to seven years and also with fine.'),
 ('CIT_4','Citizenship Act, 1951','4',
  'Citizenship by birth is governed by birth in the territory after commencement, subject to statutory exceptions including specified diplomatic and enemy alien circumstances.'),
 ('CONST_102','Constitution of Bangladesh','102',
  'The High Court Division may issue specified orders and directions to enforce fundamental rights and review unlawful public authority action, subject to constitutional conditions.')
]
LEGAL_SOURCES_MATRIX = [dict(statute_id=s, act_name=a, section_num=n, text=t,
    status='educational_summary', impacted_by_judgment=None,
    source_portal='https://bdlaws.minlaw.gov.bd', verified_official_text=False)
    for s,a,n,t in SOURCE_ROWS]
LEGAL_SOURCES_MATRIX[0].update(status='interpretation_updated_SIMULATED', impacted_by_judgment={
    'judgment_id':'DEMO_HCD_2026_01', 'court':'High Court Division (synthetic)',
    'date':'2026-01-15', 'summary':'Synthetic update: require an explicit arrest-safeguards review.',
    'is_simulated':True})

KNOWLEDGE_BASE_V1 = {'version':'1.0.0', 'as_of':'2025-12-01', 'modules':[
 {'id':'facts','required_facts':{'case_id':'present','client_name':'present'},
  'depends_on':[], 'mutually_exclusive_with':[], 'order_index':10,'linked_statutes':[],
  'template':'FACTS: {client_name} seeks relief in fictional case {case_id}.'},
 {'id':'jurisdiction','required_facts':{'forum':'magistrate'},
  'depends_on':['facts'], 'mutually_exclusive_with':[], 'order_index':20,'linked_statutes':[],
  'template':'JURISDICTION: This illustrative application is addressed to the competent Magistrate; territorial competence requires professional verification.'},
 {'id':'statutory_grounds','required_facts':{'seek_bail':True},
  'depends_on':['jurisdiction'], 'mutually_exclusive_with':['procedural_defense'], 'order_index':30,
  'linked_statutes':['CRPC_497'],
  'template':'STATUTORY GROUNDS: Request consideration of non-bailable release under [CRPC_497], subject to all statutory restrictions. No entitlement is assumed.'},
 {'id':'procedural_defense','required_facts':{'challenge_arrest':True},
  'depends_on':['jurisdiction'], 'mutually_exclusive_with':['statutory_grounds'], 'order_index':40,
  'linked_statutes':['CRPC_54','CRPC_167'],
  'template':'PROCEDURAL DEFENSE: Request examination of arrest grounds [CRPC_54] and judicial authorization of further detention [CRPC_167].'},
 {'id':'evidentiary_annex','required_facts':{'has_records':True},
  'depends_on':['procedural_defense'], 'mutually_exclusive_with':[], 'order_index':50,
  'linked_statutes':['CRPC_167'],
  'template':'EVIDENTIARY ANNEX: Attach the fictional arrest timeline and custody records for review under [CRPC_167].'},
 {'id':'prayer_for_relief','required_facts':{'request_relief':True},
  'depends_on':['jurisdiction'], 'mutually_exclusive_with':[], 'order_index':60,'linked_statutes':[],
  'template':'PRAYER: Seek an appropriate reasoned order after examination of the record and applicable law.'}
]}
# Exclusive routes are a teaching policy, not a statement that these arguments
# are legally incompatible. Real counsel can legitimately combine them.
print('Sections:', len(LEGAL_SOURCES_MATRIX), '| Modules:', len(KNOWLEDGE_BASE_V1['modules']))

Sections: 8 | Modules: 6


# 2. Deterministic Assembly Engine (Rule Evaluation & Traceability)
Assembly is a rule execution problem. A generative model could silently omit prerequisites, change a prayer or invent facts. Here, exact predicates, a validated dependency graph, explicit ordering and symmetric conflict checks determine every outcome. Missing optional Boolean facts default to false; malformed facts are rejected. Conflicting eligible strategies raise an error before any text is emitted.

The trace includes predicate results, dependency results, conflict checks, inclusion decisions and the exact knowledge-base fingerprint for every module. Output order is stable. This audit explains software behavior, not whether the legal strategy is correct.

In [2]:
class AssemblyConflictError(ValueError): pass

class DeterministicAssemblyEngine:
    BOOLEAN_FACTS = ('seek_bail','challenge_arrest','has_records','request_relief')
    def __init__(self, knowledge_base):
        self.kb = deepcopy(knowledge_base)
        self.modules = sorted(self.kb['modules'], key=lambda m:(m['order_index'],m['id']))
        self.fingerprint = digest(self.kb)
        ids = [m['id'] for m in self.modules]
        if len(ids) != len(set(ids)): raise ValueError('Duplicate module ID')
        positions = {mid:i for i,mid in enumerate(ids)}
        for m in self.modules:
            for dep in m['depends_on']:
                if dep not in positions or positions[dep] >= positions[m['id']]:
                    raise ValueError('Dependency missing, cyclic or ordered after dependent')
            for conflict in m['mutually_exclusive_with']:
                if conflict not in positions or conflict == m['id']:
                    raise ValueError('Invalid conflict reference')

    def validate(self, case_facts):
        if not isinstance(case_facts, dict): raise ValueError('case_facts must be a dictionary')
        allowed = {'case_id','client_name','forum',*self.BOOLEAN_FACTS}
        if set(case_facts)-allowed: raise ValueError('Unknown fact keys')
        f = deepcopy(case_facts)
        for key in ('case_id','client_name','forum'):
            if not isinstance(f.get(key),str) or not f[key].strip():
                raise ValueError('Required nonempty string: '+key)
            f[key] = f[key].strip()
        if f['forum'] not in ('magistrate','high_court'): raise ValueError('Unsupported forum')
        for key in self.BOOLEAN_FACTS:
            f.setdefault(key,False)
            if type(f[key]) is not bool: raise ValueError('Expected Boolean: '+key)
        return f

    def assemble(self, case_facts):
        f = self.validate(case_facts)
        evaluations = {}
        for m in self.modules:
            checks = {k: (bool(f.get(k)) if expected=='present' else f.get(k)==expected)
                      for k,expected in m['required_facts'].items()}
            evaluations[m['id']] = checks
        selected, trace = [], []
        # First resolve predicates and prerequisite closure; no text is rendered yet.
        for m in self.modules:
            checks = evaluations[m['id']]
            missing = [dep for dep in m['depends_on'] if dep not in selected]
            include = all(checks.values()) and not missing
            if include: selected.append(m['id'])
            trace.append(dict(module=m['id'], kb_version=self.kb['version'],
                kb_fingerprint=self.fingerprint, rule_results=checks,
                missing_dependencies=missing, conflict_checks=[],
                decision='included' if include else 'excluded',
                reason='rules and dependencies satisfied' if include else
                       ('predicate failed' if not all(checks.values()) else 'prerequisite excluded')))
        for m,row in zip(self.modules,trace):
            row['conflict_checks'] = [dict(other=o, active=(m['id'] in selected and o in selected))
                                     for o in sorted(m['mutually_exclusive_with'])]
        conflicts = sorted({tuple(sorted((m['id'],o))) for m in self.modules
                            if m['id'] in selected for o in m['mutually_exclusive_with'] if o in selected})
        if conflicts: raise AssemblyConflictError('Exclusive strategies: '+repr(conflicts))
        included = [m for m in self.modules if m['id'] in selected]
        text = '\n\n'.join(m['template'].format(**f) for m in included)
        citations = sorted(set(re.findall(r'\[([A-Z]+_\d+)\]',text)))
        return dict(text=text, module_sequence=selected,
                    linked_statutes=sorted({s for m in included for s in m['linked_statutes']}),
                    citations=citations, execution_trace=trace, case_facts=f,
                    kb_version=self.kb['version'], kb_as_of=self.kb['as_of'],
                    kb_fingerprint=self.fingerprint)

engine = DeterministicAssemblyEngine(KNOWLEDGE_BASE_V1)
SAMPLE_CASE = dict(case_id='DEMO-001',client_name='Amina (fictional)',forum='magistrate',
                   challenge_arrest=True,has_records=True,request_relief=True)
assembled_output = engine.assemble(SAMPLE_CASE)
print(assembled_output['text'])
table(assembled_output['execution_trace'],
      ['module','kb_version','rule_results','missing_dependencies','conflict_checks','decision','reason'])
print('KB fingerprint:', assembled_output['kb_fingerprint'])

FACTS: Amina (fictional) seeks relief in fictional case DEMO-001.

JURISDICTION: This illustrative application is addressed to the competent Magistrate; territorial competence requires professional verification.

PROCEDURAL DEFENSE: Request examination of arrest grounds [CRPC_54] and judicial authorization of further detention [CRPC_167].

EVIDENTIARY ANNEX: Attach the fictional arrest timeline and custody records for review under [CRPC_167].

PRAYER: Seek an appropriate reasoned order after examination of the record and applicable law.
module             | kb_version | rule_results                           | missing_dependencies | conflict_checks                                    | decision | reason                          
-------------------+------------+----------------------------------------+----------------------+----------------------------------------------------+----------+---------------------------------
facts              | 1.0.0      | {'case_id': True, 'client_name': 

# 3. Section-Aware RAG & AI Review Layer (Threshold-Gated Retrieval + Impact Audit)
This implementation uses sparse word TF-IDF cosine similarity rather than dense embeddings. Each indexed record prepends `[Act Name - Section N]`. Scores are lexical similarity, **not probabilities of legal correctness**. A strict `tau = 0.80` gate admits only qualifying records; otherwise the system splits explicit `and` / semicolon issues once, retrieves again and abstains on unresolved parts. This is a simple, bounded search fallback. Exact section-text queries below deliberately make the fallback reproducible; they do not demonstrate paraphrase performance. The threshold requires separate calibration on a real corpus.

The audit is independent of retrieval confidence: every mandatory citation is derived from the trusted selected module definitions, checked against actual text and against the source matrix. A newer dated judgment produces an impact alert identifying affected modules. Supplementary case-law entries are labeled synthetic research suggestions, never authorities to cite. In production these must be verified against a curated case-law database. An alert cannot decide whether a judgment applies, overrules an authority or changes the statute.

In [3]:
SUPPLEMENTARY_CASE_LAW = [dict(id='DEMO_PRECEDENT_01', linked_statutes=['CRPC_54','CRPC_167'],
    title='Synthetic research lead: arrest and remand safeguards', is_simulated=True)]

class LegalRAGReviewLayer:
    def __init__(self, sources, knowledge_base, tau=0.80):
        if not 0 <= tau <= 1: raise ValueError('Invalid threshold')
        self.sources = deepcopy(sources)
        self.kb = deepcopy(knowledge_base)
        self.by_id = {s['statute_id']:s for s in self.sources}
        if len(self.by_id) != len(self.sources): raise ValueError('Duplicate statute ID')
        self.tau = tau
        self.documents = [f"[{s['act_name']} - Section {s['section_num']}] {s['text']}" for s in self.sources]
        self.vectorizer = TfidfVectorizer(ngram_range=(1,2), lowercase=True)
        self.index = self.vectorizer.fit_transform(self.documents)

    def _search(self, query):
        scores = cosine_similarity(self.vectorizer.transform([query]), self.index)[0]
        ranked = sorted(range(len(scores)), key=lambda i:(-scores[i], self.sources[i]['statute_id']))
        accepted = [dict(statute_id=self.sources[i]['statute_id'], score=float(scores[i]),
                         context=self.documents[i]) for i in ranked if scores[i] >= self.tau]
        return dict(query=query, best_score=float(scores[ranked[0]]), accepted=accepted,
                    abstained=not bool(accepted))

    def retrieve(self, query):
        if not isinstance(query,str) or not query.strip(): raise ValueError('Empty query')
        initial = self._search(query)
        steps = [dict(stage='initial', **initial)]
        if not initial['accepted']:
            parts = [p.strip() for p in (query.split(';') if ';' in query else re.split(r'\s+and\s+',query,flags=re.I)) if p.strip()]
            if len(parts)>1:
                steps += [dict(stage='decomposed',**self._search(p)) for p in parts]
        accepted = {}
        for step in steps:
            for hit in step['accepted']:
                old = accepted.get(hit['statute_id'])
                if old is None or hit['score']>old['score']: accepted[hit['statute_id']] = hit
        return dict(tau=self.tau, steps=steps, hits=list(accepted.values()),
                    unresolved_queries=[s['query'] for s in (steps[1:] or steps) if s['abstained']],
                    answer_mode='extractive context only; no generated legal conclusion')

    def audit_pleading(self, output):
        if output['kb_fingerprint'] != digest(self.kb): raise ValueError('Unrecognized KB snapshot')
        modules = {m['id']:m for m in self.kb['modules']}
        if any(mid not in modules for mid in output['module_sequence']):
            raise ValueError('Unknown selected module')
        required = {s for mid in output['module_sequence'] for s in modules[mid]['linked_statutes']}
        actual = set(re.findall(r'\[([A-Z]+_\d+)\]',output['text']))
        alerts = []
        for sid in sorted(required-actual):
            alerts.append(dict(type='missing_mandatory_citation',statute_id=sid,
                               message='Required by selected module but absent from pleading text'))
        for sid in sorted(required|actual):
            affected = [mid for mid in output['module_sequence'] if sid in modules[mid]['linked_statutes']]
            source = self.by_id.get(sid)
            if source is None:
                alerts.append(dict(type='unknown_statute',statute_id=sid,modules=affected))
                continue
            judgment = source['impacted_by_judgment']
            if judgment and date.fromisoformat(judgment['date'])>date.fromisoformat(output['kb_as_of']):
                alerts.append(dict(type='judgment_impact',statute_id=sid,modules=affected,
                    judgment_id=judgment['judgment_id'],message=judgment['summary'],
                    simulated=judgment['is_simulated']))
        suggestions = [deepcopy(c) for c in SUPPLEMENTARY_CASE_LAW
                       if set(c['linked_statutes']) & (required|actual)]
        return dict(alerts=alerts, supplementary_case_law_suggestions=suggestions,
                    mandatory_citations=sorted(required),actual_citations=sorted(actual),
                    source_snapshot=digest(self.sources))

review = LegalRAGReviewLayer(LEGAL_SOURCES_MATRIX, KNOWLEDGE_BASE_V1)
# Two unrelated, intact section records: their joint query is below 0.80;
# each decomposed exact-record query reaches 1.0.
complex_query = review.documents[0] + '; ' + review.documents[6]
retrieval_demo = review.retrieve(complex_query)
table([dict(stage=s['stage'],best_score=f"{s['best_score']:.4f}",
            accepted=','.join(h['statute_id'] for h in s['accepted']),abstained=s['abstained'])
       for s in retrieval_demo['steps']], ['stage','best_score','accepted','abstained'])
assert retrieval_demo['steps'][0]['abstained']
assert {h['statute_id'] for h in retrieval_demo['hits']} == {'CRPC_54','CIT_4'}
assert review.retrieve('quantum neutrino spectroscopy')['hits'] == []
print('\nPleading review:')
print(json.dumps(review.audit_pleading(assembled_output),indent=2))

stage      | best_score | accepted | abstained
-----------+------------+----------+----------
initial    | 0.7749     |          | True     
decomposed | 1.0000     | CRPC_54  | False    
decomposed | 1.0000     | CIT_4    | False    

Pleading review:
{
  "alerts": [
    {
      "type": "judgment_impact",
      "statute_id": "CRPC_54",
      "modules": [
        "procedural_defense"
      ],
      "judgment_id": "DEMO_HCD_2026_01",
      "message": "Synthetic update: require an explicit arrest-safeguards review.",
      "simulated": true
    }
  ],
  "supplementary_case_law_suggestions": [
    {
      "id": "DEMO_PRECEDENT_01",
      "linked_statutes": [
        "CRPC_54",
        "CRPC_167"
      ],
      "title": "Synthetic research lead: arrest and remand safeguards",
      "is_simulated": true
    }
  ],
  "mandatory_citations": [
    "CRPC_167",
    "CRPC_54"
  ],
  "actual_citations": [
    "CRPC_167",
    "CRPC_54"
  ],
  "source_snapshot": "ffccd8efa3150c5550a691a9216dc59d8473

# 4. Regression Testing Suite on Historical Approved Cases
A living knowledge base needs fixed expected decisions before rules change. Reproducibility detects output drift; parity checks catch changed module sequences and citations; conflict checks protect strategy constraints; impact checks ensure new source judgments propagate without rebuilding pleadings.

**Provenance:** no historical files or lawyer approvals were supplied. `HISTORICAL_APPROVED_CASES` therefore contains four fictional golden fixtures with hand-written expectations and `approval_status = simulated_not_lawyer_approved`. The name follows the requested interface; it must not be represented as real approval evidence. Replace these with de-identified, genuinely approved records and documented consent before claiming historical legal parity. The 100% result below is fixture parity only, not legal accuracy.

The runner raises on failure, prints pass/fail honestly and checks actual outcomes rather than only successful process exits. The additional negative checks exercise validation, prerequisite exclusion, missing citations and low-confidence abstention.

<details>
<summary>Technical source note</summary>

Inspired by Adnan, Khurshed, Sharifeen and Momen, *JuristRAG: An Enhanced RAG System for Semantic Question Answering on Legal Statutes*, ICCIT 2025, supplied as `1921.pdf`, particularly Section III.A–C. The paper preserves statutory sections, prepends Act names and uses a similarity threshold of 0.80 with query decomposition. This notebook adapts those ideas; it does not reproduce the paper's BGE-M3, Llama, hosted generation or reported evaluation results.

</details>


In [4]:
HISTORICAL_APPROVED_CASES = [
 dict(id='H1',approval_status='simulated_not_lawyer_approved',facts=SAMPLE_CASE,
      expected_modules=['facts','jurisdiction','procedural_defense','evidentiary_annex','prayer_for_relief'],
      expected_citations=['CRPC_167','CRPC_54']),
 dict(id='H2',approval_status='simulated_not_lawyer_approved',
      facts=dict(case_id='H2',client_name='B (fictional)',forum='magistrate',seek_bail=True,request_relief=True),
      expected_modules=['facts','jurisdiction','statutory_grounds','prayer_for_relief'],expected_citations=['CRPC_497']),
 dict(id='H3',approval_status='simulated_not_lawyer_approved',
      facts=dict(case_id='H3',client_name='C (fictional)',forum='magistrate',request_relief=True,has_records=True),
      expected_modules=['facts','jurisdiction','prayer_for_relief'],expected_citations=[]),
 dict(id='H4',approval_status='simulated_not_lawyer_approved',
      facts=dict(case_id='H4',client_name='D (fictional)',forum='magistrate',challenge_arrest=True),
      expected_modules=['facts','jurisdiction','procedural_defense'],expected_citations=['CRPC_167','CRPC_54'])
]

def test_reproducibility():
    hashes = [digest(engine.assemble(SAMPLE_CASE)) for _ in range(20)]
    assert len(set(hashes)) == 1, 'Output drift'
    return '20/20 identical full-output hashes'

def test_parity():
    for fixture in HISTORICAL_APPROVED_CASES:
        out = engine.assemble(fixture['facts'])
        assert out['module_sequence']==fixture['expected_modules'], fixture['id']
        assert out['citations']==fixture['expected_citations'], fixture['id']
        assert out['linked_statutes']==fixture['expected_citations'], fixture['id']
    return '4/4 synthetic golden fixtures; 100% module/citation parity'

def test_conflicts():
    messages=[]
    for _ in range(2):
        try: engine.assemble(dict(SAMPLE_CASE,seek_bail=True))
        except AssemblyConflictError as exc: messages.append(str(exc))
        else: raise AssertionError('Conflicting strategies accepted')
    assert messages[0]==messages[1]
    # Ordering and prerequisite checks also prevent an orphan annex.
    out=engine.assemble(dict(case_id='X',client_name='X',forum='magistrate',has_records=True))
    assert 'evidentiary_annex' not in out['module_sequence']
    try: engine.assemble(dict(SAMPLE_CASE,has_records='yes'))
    except ValueError: pass
    else: raise AssertionError('Invalid Boolean accepted')
    return 'Stable conflict error; prerequisite and input validation enforced'

def test_alerting():
    baseline=deepcopy(LEGAL_SOURCES_MATRIX)
    for s in baseline: s['impacted_by_judgment']=None
    before=LegalRAGReviewLayer(baseline,KNOWLEDGE_BASE_V1)
    assert not any(a['type']=='judgment_impact' for a in before.audit_pleading(assembled_output)['alerts'])
    changed=deepcopy(baseline)
    next(s for s in changed if s['statute_id']=='CRPC_167')['impacted_by_judgment']={
        'judgment_id':'DEMO_NEW_2026_02','date':'2026-02-01','summary':'Synthetic remand review update',
        'is_simulated':True}
    after=LegalRAGReviewLayer(changed,KNOWLEDGE_BASE_V1)
    impact=[a for a in after.audit_pleading(assembled_output)['alerts'] if a['type']=='judgment_impact']
    assert len(impact)==1 and impact[0]['statute_id']=='CRPC_167'
    assert impact[0]['modules']==['procedural_defense','evidentiary_annex']
    damaged=deepcopy(assembled_output)
    damaged['text']=damaged['text'].replace('[CRPC_54]','')
    # Leaving linked_statutes metadata intact must not conceal missing text citations.
    assert any(a['type']=='missing_mandatory_citation' and a['statute_id']=='CRPC_54'
               for a in after.audit_pleading(damaged)['alerts'])
    retrieved=review.retrieve(complex_query)
    assert retrieved['steps'][0]['abstained'] and len(retrieved['hits'])==2
    assert review.retrieve('quantum neutrino spectroscopy')['hits']==[]
    return 'New judgment alerts both affected modules; missing citation and gate verified'

results=[]
for name,fn in [('1 Reproducibility',test_reproducibility),('2 Historical fixture parity',test_parity),
                ('3 Constraints and conflicts',test_conflicts),('4 RAG review alerting',test_alerting)]:
    try: results.append(dict(test=name,status='PASS',evidence=fn()))
    except Exception as exc: results.append(dict(test=name,status='FAIL',evidence=repr(exc)))
table(results,['test','status','evidence'])
assert all(r['status']=='PASS' for r in results), 'Regression failure: see summary'
print('\nScope: local prototype validated; official legal text, real judgments and lawyer approval remain unverified.')

test                        | status | evidence                                                                     
----------------------------+--------+------------------------------------------------------------------------------
1 Reproducibility           | PASS   | 20/20 identical full-output hashes                                           
2 Historical fixture parity | PASS   | 4/4 synthetic golden fixtures; 100% module/citation parity                   
3 Constraints and conflicts | PASS   | Stable conflict error; prerequisite and input validation enforced            
4 RAG review alerting       | PASS   | New judgment alerts both affected modules; missing citation and gate verified

Scope: local prototype validated; official legal text, real judgments and lawyer approval remain unverified.
